# SynthDet: Intelligent Synthetic Data Generation for Object Detection

**Problem:** Small object detection datasets lead to models that struggle with rare defect sizes, spatial regions, and edge cases. Manual annotation is expensive and slow.

**Solution:** SynthDet automatically analyzes dataset gaps and generates targeted synthetic data to fill them — improving model performance with zero manual annotation.

This notebook walks through the full workflow: **gap analysis → targeted generation → training improvement**.

> All visualizations use precomputed data — no GPU or API keys needed to view this notebook.

In [ ]:
import pickle, json, sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path(".").resolve()))
from demo_viz import *

DATA = Path("demo_data")

def load_pkl(name):
    with open(DATA / name, "rb") as f:
        return pickle.load(f)

def load_json(name):
    with open(DATA / name) as f:
        return json.load(f)

# Load all precomputed data
dataset_stats = load_pkl("dataset_stats.pkl")
strategy = load_pkl("synthesis_strategy.pkl")
sample_originals = load_pkl("sample_originals.pkl")
walkthrough = load_pkl("compositing_walkthrough.pkl")
sample_synthetics = load_pkl("sample_synthetics.pkl")
after_stats = load_pkl("after_stats.pkl")
pipeline_summary = load_json("pipeline_summary.json")
training_metrics = load_json("training_metrics.json")
profile_before = load_json("profile_before.json")
profile_after = load_json("profile_after.json")
active_learning = load_json("active_learning.json")

print("All demo data loaded.")

## 1. The Dataset

We're working with the **Laptop Scratch Detection** dataset (Roboflow) — used to train YOLO models for automated laptop grading in refurbishment workshops.

- **Format:** YOLOv12 (normalized bounding boxes)
- **Classes:** Scratch (1 class)
- **Resolution:** 860x640 px, JPEG

In [ ]:
# Key stats
s = dataset_stats
print(f"Images: {s.total_images} ({s.split_image_counts})")
print(f"Annotations: {s.total_annotations}")
print(f"Classes: {len(s.class_distributions)}")
print(f"Negative examples (no defects): {s.negative_images}  <-- critical gap!")
print(f"Unique source images: ~{s.unique_source_images} (rest are augmentation variants)")
print()

# Show 6 sample images with bboxes
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
class_names = [cd.class_name for cd in s.class_distributions]
for idx, (ax, (img, bboxes)) in enumerate(zip(axes.flat, sample_originals)):
    show_image_with_bboxes(img, bboxes, class_names, ax=ax, title=f"Sample {idx+1}")
fig.suptitle("Original Training Images", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Gap Analysis

SynthDet automatically identifies **structural weaknesses** in the dataset. These are the gaps that cause poor model performance on specific defect types.

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

# Size bucket distribution
bucket_counts = {k.value if hasattr(k, 'value') else k: v
                 for k, v in s.overall_bucket_counts.items()}
plot_size_bucket_distribution(bucket_counts, ax=ax1, title="Size Buckets", target_line=50)

# Spatial heatmap
region_counts = {k.value if hasattr(k, 'value') else k: v
                 for k, v in s.overall_region_counts.items()}
plot_spatial_heatmap(region_counts, ax=ax2, title="Spatial Distribution")

# Annotations per image
plot_annotations_per_image(s.annotations_per_image_histogram, ax=ax3)

plt.tight_layout()
plt.show()

print(f"Size bucket uniformity: {s.bucket_uniformity:.2f} (1.0 = perfectly uniform)")
print(f"Spatial uniformity:     {s.region_uniformity:.2f}")
print(f"Negative examples:      {s.negative_images} (target: ~15% of dataset)")

## 3. Synthesis Strategy

Based on the gaps identified above, SynthDet creates a **prioritized generation plan**. Tasks are ranked by severity — zero-representation gaps get highest priority.

In [ ]:
fig, ax = plt.subplots(figsize=(14, max(4, len(strategy.generation_tasks) * 0.5)))
plot_generation_tasks(strategy.generation_tasks, ax=ax)
plt.tight_layout()
plt.show()

print(f"Target dataset size: {strategy.target_total_images} images")
print(f"Planned synthetic:   {strategy.total_synthetic_images} images")
print(f"Generation tasks:    {len(strategy.generation_tasks)}")
print(f"Negative ratio:      {strategy.negative_ratio:.0%}")

## 4. Compositor Pipeline

The compositor is SynthDet's most reliable generation method:

1. **Extract** defect patches from existing annotated images
2. **Inpaint** original images to create clean backgrounds
3. **Composite** patches onto backgrounds with Poisson blending
4. **Annotations are free** — we know exactly where we placed each defect

No detection model needed. No annotation errors. Deterministic and reproducible.

In [ ]:
n_walk = len(walkthrough)
fig, axes = plt.subplots(n_walk, 3, figsize=(14, 4.5 * n_walk))
if n_walk == 1:
    axes = axes[np.newaxis, :]

class_names = [cd.class_name for cd in dataset_stats.class_distributions]
col_titles = ["Original (annotated)", "Clean Background", "Synthetic Result"]

for row, ((orig_img, orig_bboxes), bg_img, (syn_img, syn_bboxes)) in enumerate(walkthrough):
    show_image_with_bboxes(orig_img, orig_bboxes, class_names, ax=axes[row, 0])
    axes[row, 1].imshow(bg_img)
    axes[row, 1].axis("off")
    show_image_with_bboxes(syn_img, syn_bboxes, class_names, ax=axes[row, 2])
    if row == 0:
        for col, title in enumerate(col_titles):
            axes[row, col].set_title(title, fontsize=11, fontweight="bold")

fig.suptitle("Compositing Walkthrough", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 5. Dataset Improvement

After generation, the structural gaps are filled. Compare the before/after distributions:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Before buckets
before_buckets = {k.value if hasattr(k, 'value') else k: v
                  for k, v in dataset_stats.overall_bucket_counts.items()}
plot_size_bucket_distribution(before_buckets, ax=axes[0], title="Buckets: Before")

# After buckets
after_buckets = {k.value if hasattr(k, 'value') else k: v
                 for k, v in after_stats.overall_bucket_counts.items()}
plot_size_bucket_distribution(after_buckets, ax=axes[1], title="Buckets: After")

# Before spatial
before_regions = {k.value if hasattr(k, 'value') else k: v
                  for k, v in dataset_stats.overall_region_counts.items()}
plot_spatial_heatmap(before_regions, ax=axes[2], title="Spatial: Before")

# After spatial
after_regions = {k.value if hasattr(k, 'value') else k: v
                 for k, v in after_stats.overall_region_counts.items()}
plot_spatial_heatmap(after_regions, ax=axes[3], title="Spatial: After")

plt.tight_layout()
plt.show()

print(f"Bucket uniformity: {dataset_stats.bucket_uniformity:.2f} -> {after_stats.bucket_uniformity:.2f}")
print(f"Spatial uniformity: {dataset_stats.region_uniformity:.2f} -> {after_stats.region_uniformity:.2f}")
print(f"Total images: {dataset_stats.total_images} -> {after_stats.total_images}")
print(f"Total annotations: {dataset_stats.total_annotations} -> {after_stats.total_annotations}")

## 6. Training Impact

We trained YOLOv8n on the original dataset (baseline) and the augmented dataset. The synthetic data significantly improves detection across all defect sizes.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Training curve
plot_training_curve(training_metrics, ax=ax1)

# Per-bucket recall comparison
plot_bucket_recall_comparison(
    profile_before["per_bucket_map"],
    profile_after["per_bucket_map"],
    ax=ax2,
)

plt.tight_layout()
plt.show()

print(f"Baseline mAP50:   {profile_before['overall_map50']:.2f}")
print(f"Augmented mAP50:  {profile_after['overall_map50']:.2f}")
print(f"Improvement:      +{profile_after['overall_map50'] - profile_before['overall_map50']:.2f}")

## 7. Active Learning

SynthDet supports iterative improvement: **generate → train → evaluate → refine strategy → repeat**.

Each iteration targets the model's current failure modes, producing diminishing but meaningful returns.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Active learning progress
plot_active_learning_progress(active_learning, ax=ax1)

# Final region recall heatmap
plot_region_recall_heatmap(
    profile_after["per_region_map"],
    ax=ax2,
    title="Final Per-Region Recall",
)

plt.tight_layout()
plt.show()

## 8. Get Started

### CLI Commands

```bash
# Analyze your dataset
uv run python -m synthdet.analysis data/data.yaml

# Generate synthetic data (compositor)
uv run python -m synthdet.pipeline data/data.yaml --output output/ --method compositor --seed 42

# Full pipeline with training and active learning
uv run python -m synthdet.pipeline data/data.yaml --output output/ --active-learning 3 --seed 42
```

### Generation Methods

| Method | Annotation | Cost | Quality |
|--------|-----------|------|---------|
| **Compositor** | Perfect (by construction) | Free | Deterministic |
| **Inpainting** (Imagen 3) | Perfect (mask-based) | ~$0.04/img | High variety |
| **Generative Compositor** | Perfect (by construction) | ~$0.04/img | API patches + blending |
| **Modify & Annotate** | Auto-detected (~70%) | ~$0.08/img | Whole-image transform |

### Install

```bash
pip install -e ".[generation,annotation,embeddings,quality]"
```

In [ ]:
# Summary
ps = pipeline_summary
print("=" * 50)
print("  SynthDet Pipeline Summary")
print("=" * 50)
print(f"  Original dataset:   {ps['original_count']} images")
print(f"  Synthetic added:    {ps['synthetic_count']} images")
print(f"  Augmented dataset:  {ps['train_count']} train / {ps['valid_count']} valid / {ps['test_count']} test")
print(f"  Generation cost:    ${ps['cost']:.2f}")
print(f"  Methods used:       {', '.join(ps['methods'])}")
print()
print(f"  Baseline mAP50:     {profile_before['overall_map50']:.2f}")
print(f"  Final mAP50:        {profile_after['overall_map50']:.2f}")
improvement = profile_after['overall_map50'] - profile_before['overall_map50']
print(f"  Improvement:        +{improvement:.2f} ({improvement/profile_before['overall_map50']:.0%} relative)")
print("=" * 50)